# Data Preparation

## Pinyin Mapping

In [1]:
INITIALS = [
    "", "b","p","m","f","d","t","n","l","g","k","h",
    "j","q","x","zh","ch","sh","r","z","c","s"
]

FINALS = [
    "a","ai","an","ang","ao","e","ei","en","eng","er",
    "i","ia","ian","iang","iao","ie","in","ing","iong","iu",
    "o","ong","ou",
    "u","ua","uai","uan","uang","ui","un","uo",
    "v","ve","van","vn"
]

TONES = ["1","2","3","4","5"]

ZERO_INITIAL_MAP = {
    # y-series
    "yi":"i",
    "ya":"ia",
    "yao":"iao",
    "ye":"ie",
    "you":"iu",
    "yan":"ian",
    "yin":"in",
    "yang":"iang",
    "ying":"ing",
    "yong":"iong",

    "yu":"v",
    "yue":"ve",
    "yuan":"van",
    "yun":"vn",

    # w-series
    "wu":"u",
    "wa":"ua",
    "wo":"uo",
    "wai":"uai",
    "wei":"ui",
    "wan":"uan",
    "wen":"un",
    "wang":"uang",
    "weng":"ong",
}


def normalize_zero_initial(base):
    return ZERO_INITIAL_MAP.get(base, base)


def normalize_jqx(final):
    if final.startswith("u"):
        mapping = {
            "u": "v",
            "ue": "ve",
            "uan": "van",
            "un": "vn",
        }
        return mapping.get(final, final)
    return final


init2idx = {v:i for i,v in enumerate(INITIALS)}
final2idx = {v:i for i,v in enumerate(FINALS)}
tone2idx = {v:i for i,v in enumerate(TONES)}

idx2init = {i:v for v,i in init2idx.items()}
idx2final = {i:v for v,i in final2idx.items()}
idx2tone = {i:v for v,i in tone2idx.items()}


def decompose_pinyin(token: str):
    tone = token[-1]

    if not tone.isdigit():
        tone = "5" # neutral tone, usually has no digit
        base = token
    else:
        base = token[:-1]

    base = normalize_zero_initial(base)

    # longest-match for initials

    try:
        for ini in sorted(INITIALS, key=len, reverse=True):
            if base.startswith(ini):
                final = base[len(ini):]

                if ini in {"j","q","x"}:
                    final = normalize_jqx(final)
                
                return [init2idx[ini], final2idx[final], tone2idx[tone]]
    except Exception:
        raise ValueError("Invalid pinyin token")

## Dataset: Text + Pinyin

In [2]:
import pandas as pd

dataset = pd.read_csv("/kaggle/input/notebooks/davidvista/pinyin-dataset-labelling/chinese-short-sentences-pinyin.csv")

In [3]:
dataset.head()

,text,pinyin
0,一个特殊群体的期待在密切关注此次特金会的人中有一个特殊的群体在日本殖民期间来到日本的韩国朝鲜...,yi1 ge4 te4 shu1 qun2 ti3 de qi1 dai4 zai4 mi4...
1,他们希望政治解冻有助于让朝鲜摆脱孤立,ta1 men xi1 wang4 zheng4 zhi4 jie3 dong4 you3 ...
2,他认为对这个奖的妄想可能会促使双方做出轻率的承诺但也可能有助于艰巨的和平进程,ta1 ren4 wei2 dui4 zhe4 ge4 jiang3 de wang4 xi...
3,欢迎阅读纪思道文章的中文版,huan1 ying2 yue4 du2 ji4 si1 dao4 wen2 zhang1 ...
4,不法业者向一些中国女性编造了一个美好的未来到美国开始新生活并拥有一份合法的工作,bu4 fa3 ye4 zhe3 xiang4 yi1 xie1 zhong1 guo2 n...


## Dataset: Pinyin Embeddings

In [4]:
import pickle as pkl

with open("/kaggle/input/notebooks/davidvista/mandarin-sounds-dataset/pinyin_embeddings.pkl", "rb") as f:
    embedding_map = pkl.load(f)

In [5]:
embedding_map['wo3'].shape

(1024,)

## Training Dataset

First, filter the sequences where at least one token is missing a sound (only bo token is expected, with around 70 sentences):

In [6]:
def align(token):
    if token[-1].isalpha():  # e.g. de -> de5
        return token + '5'
    return token

# To see filtered values
dataset[
    dataset['pinyin'].str.split().transform(lambda x: any(align(y) not in embedding_map.keys() for y in x))
]

,text,pinyin
1267,纽约时报现年八十二岁的阿尔及利亚強人总统阿卜杜勒阿齐兹布特弗利卡月底将下台,niu3 yue1 shi2 bao4 xian4 nian2 ba1 shi2 er4 s...
1369,如今安倍的政治对手已经磨刀霍霍安倍经济学命运未卜,ru2 jin1 an1 bei4 de zheng4 zhi4 dui4 shou3 yi...
1440,纽约时报马尔代夫总统易卜拉欣穆罕默德萨利赫所在的政党似乎将在该国议会选举中获得压倒性胜利,niu3 yue1 shi2 bao4 ma3 er3 dai4 fu1 zong3 ton...
2203,两个家庭的命运以悲剧性的方式交汇了波尔森家失去了三个年幼的孩子炸死他们的是易卜拉欣家的两个儿子,liang3 ge4 jia1 ting2 de ming4 yun4 yi3 bei1 j...
4805,阿卜杜拉赫卜表示自己和前夫曾因此收到死亡威胁她希望公开身份能够让自己和家人免于被报复,a1 bo du4 la1 he4 bo biao3 shi4 zi4 ji3 he2 qi...
...,...,...
152505,营地在一条路边那是通往夏河县及著名的庞大藏传佛教寺院拉卜楞寺的道路,ying2 di4 zai4 yi1 tiao2 lu4 bian1 na4 shi4 to...
152511,这两位僧人曾在拉卜楞寺住过寺里的僧人赶来这里迎接他们,zhe4 liang3 wei4 seng1 ren2 ceng2 zai4 la1 bo ...
157326,如果我们收不到货东西就会更贵甚至可能要关门卖黄瓜萝卜和西红柿的左启超音说,ru2 guo3 wo3 men shou1 bu4 dao4 huo4 dong1 xi1...
157327,在他说话的同时一个女人指责他胡来提高萝卜的价格,zai4 ta1 shuo1 hua4 de tong2 shi2 yi1 ge4 nv3 ...


In [7]:
# Actual filtration
dataset = dataset[
    dataset['pinyin'].str.split().transform(lambda x: all(align(y) in embedding_map.keys() for y in x))
]

In [8]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 161392 entries, 0 to 161463
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    161392 non-null  object
 1   pinyin  161392 non-null  object
dtypes: object(2)
memory usage: 3.7+ MB


Next, pre-define the index of pinyin tokens for embedding lookup:

In [9]:
import numpy as np
import torch

all_pinyin_tokens = list(embedding_map.keys())

pinyin2idx = {tok: i for i, tok in enumerate(all_pinyin_tokens)}
embedding_matrix = torch.tensor(np.stack([embedding_map[tok] for tok in all_pinyin_tokens]), dtype=torch.float32)

Build vocabulary for decoder:

In [10]:
from collections import Counter

def build_char_vocab(df, min_freq=1, add_pad=True, add_unk=True):
    """
    Build character vocabulary from the 'text' column.
    Returns char2idx and idx2char.
    """
    counter = Counter()
    for text in df['text']:
        counter.update(text)
    # Filter by min_freq if needed
    chars = [ch for ch, cnt in counter.items() if cnt >= min_freq]
    chars = sorted(chars)
    vocab = []
    if add_pad:
        vocab.append('<PAD>')
    if add_unk:
        vocab.append('<UNK>')
    vocab.extend(chars)
    char2idx = {ch: i for i, ch in enumerate(vocab)}
    idx2char = vocab
    return char2idx, idx2char

char2idx, idx2char = build_char_vocab(dataset)
print(f"Vocabulary size: {len(char2idx)}")

Vocabulary size: 5464


In [11]:
def prepare_bert_dataset(df, tokenizer, char2idx, decompose_pinyin, pinyin2idx, align, max_length=512, ignore_idx=-100):
    """
    Prepares a dataset for BERT fine‑tuning on pinyin prediction and character prediction.

    Args:
        df: pandas DataFrame with columns 'text' and 'pinyin'
        tokenizer: BertTokenizer
        char2idx: dict mapping character to integer index
        decompose_pinyin: function (pinyin_str) -> [init_idx, final_idx, tone_idx]
        pinyin2idx: dict mapping pinyin token to embedding index
        align: function to canonicalize pinyin token
        max_length: maximum sequence length after adding [CLS] and [SEP]
        ignore_idx: value to use for padding (default -100)

    Returns:
        data: dict with keys:
            'input_ids', 'attention_mask', 'pinyin_components', 'pinyin_idxs', 'char_idxs'
    """
    input_ids_list = []
    attention_mask_list = []
    pinyin_components_list = []
    pinyin_idxs_list = []
    char_idxs_list = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Tokenizing"):
        text = row['text']
        pinyin_str = row['pinyin']

        encoding = tokenizer(
            text,
            max_length=max_length,
            truncation=True,
            return_offsets_mapping=True,
            padding=False,
            return_tensors=None
        )
        input_ids = encoding['input_ids']
        attention_mask = encoding['attention_mask']
        offset_mapping = encoding['offset_mapping']

        # Filter out special tokens ([CLS], [SEP]) and padding using offset mapping
        filtered_input_ids = []
        filtered_attention_mask = []
        # We'll also collect character indices in the same order
        filtered_char_idxs = []

        # The offset mapping: each token has (start, end) character indices in the original string.
        # We skip tokens that cover no character (start == end == 0).
        for i, (start, end) in enumerate(offset_mapping):
            if start == 0 and end == 0:
                continue
            ch = text[start:start+1]  # assume single character
            char_idx = char2idx.get(ch, char2idx.get('<UNK>', 0))
            filtered_input_ids.append(input_ids[i])
            filtered_attention_mask.append(attention_mask[i])
            filtered_char_idxs.append(char_idx)

        # Check alignment
        if len(filtered_input_ids) != len(text):
            # Some characters may be split into multiple tokens; we could average later, but here we skip.
            print(f"Skipping sample {idx}: token count {len(filtered_input_ids)} != character count {len(text)}")
            continue

        # Pinyin tokens
        pinyin_tokens = pinyin_str.split()
        if len(pinyin_tokens) != len(text):
            print(f"Skipping sample {idx}: pinyin token count {len(pinyin_tokens)} != character count {len(text)}")
            continue

        pinyin_component_idxs = []
        pinyin_idxs = []
        skip_sample = False
        for token in pinyin_tokens:
            try:
                comp = decompose_pinyin(token)
                pinyin_component_idxs.append(comp)
                pinyin_idxs.append(pinyin2idx[align(token)])
            except Exception as e:
                print(f"Skipping sample {idx} due to pinyin error: {token} - {e}")
                skip_sample = True
                break
        if skip_sample:
            continue

        input_ids_list.append(filtered_input_ids)
        attention_mask_list.append(filtered_attention_mask)
        pinyin_components_list.append(pinyin_component_idxs)
        pinyin_idxs_list.append(pinyin_idxs)
        char_idxs_list.append(filtered_char_idxs)

    return {
        'input_ids': input_ids_list,
        'attention_mask': attention_mask_list,
        'pinyin_components': pinyin_components_list,
        'pinyin_idxs': pinyin_idxs_list,
        'char_idxs': char_idxs_list
    }

In [12]:
import torch
from torch.utils.data import Dataset
from transformers import BertTokenizer
from tqdm import tqdm


class BERTPinyinDataset(Dataset):
    def __init__(self, data_dict):
        self.input_ids = data_dict['input_ids']
        self.attention_mask = data_dict['attention_mask']
        self.pinyin_components = data_dict['pinyin_components']
        self.pinyin_idxs = data_dict['pinyin_idxs']
        self.char_idxs = data_dict['char_idxs']

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.input_ids[idx], dtype=torch.long),
            'attention_mask': torch.tensor(self.attention_mask[idx], dtype=torch.long),
            'pinyin_components': torch.tensor(self.pinyin_components[idx], dtype=torch.long),
            'pinyin_idxs': torch.tensor(self.pinyin_idxs[idx], dtype=torch.long),
            'char_idxs': torch.tensor(self.char_idxs[idx], dtype=torch.long)
        }

In [13]:
tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
data = prepare_bert_dataset(dataset, tokenizer, char2idx, decompose_pinyin, pinyin2idx, align)
dataset = BERTPinyinDataset(data)

Tokenizing: 100%|██████████| 161392/161392 [00:59<00:00, 2693.83it/s]


Create dataloaders:

In [15]:
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch

def collate_fn_bert(batch, ignore_idx=-100):
    input_ids = [item['input_ids'] for item in batch]
    attention_masks = [item['attention_mask'] for item in batch]
    pinyin_comp = [item['pinyin_components'] for item in batch]
    pinyin_idxs = [item['pinyin_idxs'] for item in batch]
    char_idxs = [item['char_idxs'] for item in batch]

    input_ids_padded = nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=0)
    attention_mask_padded = nn.utils.rnn.pad_sequence(attention_masks, batch_first=True, padding_value=0)
    pinyin_comp_padded = nn.utils.rnn.pad_sequence(pinyin_comp, batch_first=True, padding_value=ignore_idx)
    pinyin_idxs_padded = nn.utils.rnn.pad_sequence(pinyin_idxs, batch_first=True, padding_value=ignore_idx)
    char_idxs_padded = nn.utils.rnn.pad_sequence(char_idxs, batch_first=True, padding_value=ignore_idx)
    char_lengths = torch.tensor([len(c) for c in char_idxs], dtype=torch.long)

    return (input_ids_padded, attention_mask_padded, pinyin_comp_padded,
            pinyin_idxs_padded, char_idxs_padded, char_lengths)

In [16]:
def create_bert_dataloaders(dataset, batch_size, val_split=0.1, test_split=0.1, pad_idx_components=-100, shuffle=True):
    """
    Split the BERT pinyin dataset into train, validation, test and return DataLoaders.
    Args:
        dataset: BERTPinyinDataset instance
        batch_size: int
        val_split: float (fraction for validation)
        test_split: float (fraction for test)
        pad_idx_components: int, padding value for pinyin targets
        shuffle: bool, whether to shuffle training data
    Returns:
        train_loader, val_loader, test_loader
    """
    total = len(dataset)
    val_size = int(total * val_split)
    test_size = int(total * test_split)
    train_size = total - val_size - test_size
    
    assert train_size > 0, "Not enough data for training"
    assert val_size >= 0 and test_size >= 0, "Split sizes must be non-negative"
    
    train_dataset, val_dataset, test_dataset = random_split(
        dataset, [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=shuffle,
        collate_fn=lambda b: collate_fn_bert(b, pad_idx_components)
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=lambda b: collate_fn_bert(b, pad_idx_components)
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=lambda b: collate_fn_bert(b, pad_idx_components)
    )
    
    return train_loader, val_loader, test_loader

In [17]:
train_loader, val_loader, test_loader = create_bert_dataloaders(dataset, batch_size=32)

# Model Preparation

## Encoder

In [18]:
from transformers import BertModel, BertTokenizer
import torch
import torch.nn as nn

# Device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Load BERT tokenizer and model
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")
bert_model = BertModel.from_pretrained("bert-base-chinese")

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
import torch.nn as nn

class ClassifierHead(nn.Module):
    """MLP with one hidden layer for classification."""
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [20]:
class BertPinyinEncoder(nn.Module):
    def __init__(self, bert_model, num_initials, num_finals, num_tones, mlp_hidden_dim=256, dropout=0.3):
        super().__init__()
        self.bert = bert_model
        self.initial_head  = ClassifierHead(768, mlp_hidden_dim, num_initials, dropout)
        self.final_head    = ClassifierHead(768, mlp_hidden_dim, num_finals, dropout)
        self.tone_head     = ClassifierHead(768, mlp_hidden_dim, num_tones, dropout)
        self.phonetic_head = ClassifierHead(768, mlp_hidden_dim, 1024, dropout)

    def forward(self, input_ids, attention_mask, return_hidden=False):
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state   # (batch, seq_len, 768)
        logits_i = self.initial_head(hidden)
        logits_f = self.final_head(hidden)
        logits_t = self.tone_head(hidden)

        if return_hidden:
            phonetic = self.phonetic_head(hidden)
            return logits_i, logits_f, logits_t, phonetic
        return logits_i, logits_f, logits_t

## Decoder

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class Decoder(nn.Module):
    """
    Decodes character‑level hidden representations (from an encoder) into character indices.
    Optionally applies a BiLSTM on top of the input hidden states.
    """
    def __init__(self, input_dim, hidden_dim, num_layers, vocab_size_char, mlp_hidden_dim=128, dropout=0.3):
        """
        Args:
            input_dim: dimension of input hidden states (e.g., encoder hidden_dim * 2)
            hidden_dim: LSTM hidden size
            num_layers: number of LSTM layers
            vocab_size_char: size of character vocabulary
            mlp_hidden_dim: hidden dimensionality of MLP classfication head
            dropout: dropout probability (applied after LSTM)
        """
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_dim, num_layers,
            bidirectional=True, batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.head = ClassifierHead(1024, mlp_hidden_dim, vocab_size_char, dropout)

    def forward(self, hidden_states, char_lengths):
        """
        Args:
            hidden_states: (batch, max_chars, input_dim) - padded character‑level representations
            char_lengths: (batch,) - actual lengths of each sequence (for packing)
        Returns:
            logits: (batch, max_chars, vocab_size_char) - unnormalized scores for each character
        """
        # Pack for variable‑length sequences
        packed = pack_padded_sequence(
            hidden_states, char_lengths.cpu(),
            batch_first=True, enforce_sorted=False
        )
        lstm_out, _ = self.lstm(packed)
        lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True)
        lstm_out = self.dropout(lstm_out)
        logits = self.head(lstm_out)
        return logits

## Encoder-Decoder Pipeline

In [22]:
class BertEncoderDecoderPipeline(nn.Module):
    def __init__(self, encoder, decoder, freeze_encoder=False):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.freeze_encoder = freeze_encoder
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask, char_lengths, return_encoder_output=False):
        logits_i, logits_f, logits_t, hidden = self.encoder(input_ids, attention_mask, return_hidden=True)
        char_logits = self.decoder(hidden, char_lengths)
        if return_encoder_output:
            return char_logits, (logits_i, logits_f, logits_t, hidden)
        return char_logits

    def get_hidden(self, input_ids, attention_mask, char_lengths):
        *_, hidden = self.encoder(input_ids, attention_mask, return_hidden=True)
        return hidden

# Training Stage

## Frozen Encoder

In [23]:
# Instantiate model
num_initials = len(init2idx)
num_finals   = len(final2idx)
num_tones    = len(tone2idx)
encoder = BertPinyinEncoder(bert_model, num_initials, num_finals, num_tones).to(device)

In [24]:
# Load the saved state dict
encoder.load_state_dict(torch.load('/kaggle/input/datasets/davidvista/mandarin-phonetics-bert/bert_pinyin_best.pt', map_location=device))
encoder.to(device)

print("Loaded encoder successfully!")

Loaded encoder successfully!


In [25]:
input_dim = 1024
hidden_dim = 512
mlp_hidden_dim = 256
num_layers = 4
vocab_size_char = len(idx2char)

decoder = Decoder(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    vocab_size_char=vocab_size_char,
    mlp_hidden_dim=mlp_hidden_dim
).to(device)

In [26]:
# Create pipeline with frozen encoder
pipeline = BertEncoderDecoderPipeline(encoder, decoder, freeze_encoder=True)
pipeline.to(device)

print("Created pipeline successfully")

Created pipeline successfully


In [27]:
# Loss function for character prediction (ignores padding)
criterion_char = nn.CrossEntropyLoss(ignore_index=-100)

# Optimizer for decoder only (since encoder is frozen)
optimizer = torch.optim.AdamW(pipeline.decoder.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

# Training parameters
num_epochs = 10
patience = 3
best_val_loss = float('inf')
counter = 0

In [28]:
import pickle
from tqdm import tqdm
import torch.nn as nn

# Training loop with logging
best_val_loss = float('inf')
counter = 0
metrics_log = []

for epoch in range(num_epochs):
    # --- Training ---
    pipeline.train()
    total_train_loss = 0
    train_steps = 0
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')
    for batch in train_pbar:
        # Unpack batch: input_ids, attention_mask, pinyin_comp, pinyin_idxs, char_idxs, char_lengths
        input_ids, attention_mask, _, _, char_idxs, char_lengths = batch
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        char_idxs = char_idxs.to(device)
        char_lengths = char_lengths.to(device)

        # Forward pass through pipeline (returns char_logits only)
        char_logits = pipeline(input_ids, attention_mask, char_lengths)

        # Compute character loss (ignore padding)
        loss = criterion_char(char_logits.reshape(-1, vocab_size_char), char_idxs.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        train_steps += 1
        train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_train_loss = total_train_loss / train_steps

    # --- Validation ---
    pipeline.eval()
    total_val_loss = 0
    val_steps = 0
    total_correct = 0
    total_chars = 0
    val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]')
    with torch.no_grad():
        for batch in val_pbar:
            input_ids, attention_mask, _, _, char_idxs, char_lengths = batch
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            char_idxs = char_idxs.to(device)
            char_lengths = char_lengths.to(device)

            char_logits = pipeline(input_ids, attention_mask, char_lengths)
            loss = criterion_char(char_logits.reshape(-1, vocab_size_char), char_idxs.reshape(-1))
            total_val_loss += loss.item()
            val_steps += 1

            preds = char_logits.argmax(dim=-1)
            mask = char_idxs != -100   # ignore padding
            correct = (preds == char_idxs) & mask
            total_correct += correct.sum().item()
            total_chars += mask.sum().item()

            val_pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_val_loss = total_val_loss / val_steps
    val_acc = total_correct / total_chars if total_chars > 0 else 0
    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}, Val Acc = {val_acc:.4f}")

    metrics_log.append({
        'epoch': epoch + 1,
        'train_loss': avg_train_loss,
        'val_loss': avg_val_loss,
        'val_acc': val_acc
    })

    # Early stopping and model saving
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        # Save the whole pipeline or just the decoder
        torch.save(pipeline.decoder.state_dict(), 'bert_decoder_best.pt')
        print(f"  -> Best model saved (val loss: {best_val_loss:.4f})")
        counter = 0
    else:
        counter += 1
        print(f"  -> No improvement for {counter} epoch(s).")
        if counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

    scheduler.step(avg_val_loss)

print("Training complete.")

with open('bert_decoder_metrics.pkl', 'wb') as f:
    pickle.dump(metrics_log, f)

Epoch 1/10 [Val]: 100%|██████████| 505/505 [01:08<00:00,  7.34it/s, loss=0.4328]


Epoch 1: Train Loss = 1.5428, Val Loss = 0.3927, Val Acc = 0.8810
  -> Best model saved (val loss: 0.3927)


Epoch 2/10 [Val]: 100%|██████████| 505/505 [01:08<00:00,  7.35it/s, loss=0.2607]


Epoch 2: Train Loss = 0.4551, Val Loss = 0.2501, Val Acc = 0.9237
  -> Best model saved (val loss: 0.2501)


Epoch 3/10 [Val]: 100%|██████████| 505/505 [01:08<00:00,  7.33it/s, loss=0.1610]


...

Epoch 8: Train Loss = 0.1704, Val Loss = 0.1132, Val Acc = 0.9675
  -> Best model saved (val loss: 0.1132)


Epoch 9/10 [Val]: 100%|██████████| 505/505 [01:09<00:00,  7.32it/s, loss=0.0676]


Epoch 9: Train Loss = 0.1592, Val Loss = 0.1071, Val Acc = 0.9690
  -> Best model saved (val loss: 0.1071)


Epoch 10/10 [Val]: 100%|██████████| 505/505 [01:09<00:00,  7.29it/s, loss=0.0668]


Epoch 10: Train Loss = 0.1502, Val Loss = 0.1021, Val Acc = 0.9709
  -> Best model saved (val loss: 0.1021)
Training complete.


## Evaluate Performance

In [29]:
def evaluate_bert_pipeline(pipeline, dataloader, device, ignore_index=-100):
    """
    Evaluate the BERT-based pipeline on a dataloader.
    Returns character accuracy over non‑padded characters.
    """
    pipeline.eval()
    total_correct = 0
    total_chars = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluation'):
            # Unpack batch: input_ids, attention_mask, pinyin_comp, pinyin_idxs, char_idxs, char_lengths
            input_ids, attention_mask, _, _, char_idxs, char_lengths = batch
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            char_idxs = char_idxs.to(device)
            char_lengths = char_lengths.to(device)

            # Forward pass
            char_logits = pipeline(input_ids, attention_mask, char_lengths)
            preds = char_logits.argmax(dim=-1)

            mask = char_idxs != ignore_index
            correct = (preds == char_idxs) & mask
            total_correct += correct.sum().item()
            total_chars += mask.sum().item()

    accuracy = total_correct / total_chars if total_chars > 0 else 0
    return accuracy

In [30]:
# Load the best encoder from stage 1
encoder.load_state_dict(torch.load('/kaggle/input/datasets/davidvista/mandarin-phonetics-bert/bert_pinyin_best.pt', map_location=device))
encoder.eval()
print("Loaded encoder successfully")

Loaded encoder successfully


In [31]:
# Load the best decoder
decoder.load_state_dict(torch.load('bert_decoder_best.pt', map_location=device))
decoder.eval()
print("Loaded decoder successfully")

Loaded decoder successfully


In [32]:
# Create pipeline
pipeline = BertEncoderDecoderPipeline(encoder, decoder, freeze_encoder=True)
pipeline.to(device)
pipeline.eval()

print("Pipeline loaded successfully for evaluation.")

Pipeline loaded successfully for evaluation.


In [33]:
test_acc = evaluate_bert_pipeline(pipeline, test_loader, device, ignore_index=-100)
print(f"Final test accuracy: {test_acc:.4f}")

Evaluation: 100%|██████████| 505/505 [01:10<00:00,  7.19it/s]

Final test accuracy: 0.9725
